# Lohoff Figure 2a recreation with gastrulation-atlas data

Run TissueAgent to recreate the **Gastrulation atlas** and **seqFISH** panels from the authors' published joint UMAP coordinates and cell-type labels. The inputs contain 16,861 atlas cells and 57,536 seqFISH cells; excluding `Low quality` cells leaves 16,861 and 52,568 plotted cells, respectively.

The two CSV files were extracted from [`MGA_joint_UMAP.Rds`](https://content.cruk.cam.ac.uk/jmlab/SpatialMouseAtlas2020/MGA_joint_UMAP.Rds), released with [Lohoff et al.](https://doi.org/10.1038/s41587-021-01006-2). Sources and checksums are recorded in `data/provenance_lohoff-2a.json`. These are **processed coordinates and annotations, not raw atlas expression**; this demo reconstructs the plotting workflow, not the upstream integration or UMAP computation.

The original seqFISH AnnData is retained as additional input. Its `obsm['X_umap']` differs from the published joint embedding, so both panels use the matching CSV coordinates. No color-map file is supplied; any colors estimated from the reference must be documented.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import socket
import subprocess
import sys
from datetime import datetime, timezone

from IPython.display import Image, display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "pyproject.toml").exists(), "Run this notebook from demo/ or the repository root."

TASK_NAME = "figure_recreation_lohoff-2a_with_atlas"
TASK_ID = "lohoff_2a_published_joint_atlas_seqfish"
MODEL = "gpt-5.5"
DATA_DIR = REPO_ROOT / "demo/data"
DATASETS = [
    DATA_DIR / "dataset_lohoff_et_al_seqfish.h5ad",
    DATA_DIR / "gastrulation_atlas_joint_umap.csv",
    DATA_DIR / "seqfish_author_joint_umap.csv",
]
REFERENCE = DATA_DIR / "reference_lohoff-2a.png"
ATTACHMENTS = [REFERENCE, DATA_DIR / "instructions_lohoff-2a.md"]
INPUTS = [*DATASETS, *ATTACHMENTS]
OUTPUT_DIR = REPO_ROOT / "demo/outputs" / TASK_NAME
PROMPT = "I have provided the seqFISH dataset dataset_lohoff_et_al_seqfish.h5ad and the matching published gastrulation-atlas data for Lohoff et al. Fig. 2a. The files gastrulation_atlas_joint_umap.csv and seqfish_author_joint_umap.csv contain the authors' joint UMAP coordinates (UMAP1, UMAP2), cell IDs, population, embryo, and cell-type labels. Create a side-by-side UMAP figure with panels titled Gastrulation atlas and seqFISH, colored consistently by cell type and with shared UMAP1 and UMAP2 labeling. A reference image is included in reference_lohoff-2a.png. Make the supported plot content and overall layout as close as possible to the reference.\n\nUse the genuine joint coordinates in BOTH CSVs for this figure. The original demo AnnData is retained as additional seqFISH input, but its obsm['X_umap'] is a different embedding and must not be mixed with the atlas joint coordinates. The author figure excludes cells labeled Low quality; document all exclusions and report the number of plotted cells in each panel. Do not fabricate cells, recompute the supplied embedding, modify individual coordinates to resemble the reference, or paste reference-image pixels into the result. An explicitly documented axis-direction change for visual orientation is acceptable. No color-map file is supplied; document any approximated colors.\n\nSave the recreated figure as PNG and PDF, executable plotting code, the plotted cell-level data, a color map, and a brief reproduction report under the project's outputs directory. Identify the data sources and clarify that this is figure reproduction from published processed coordinates, not a rerun of raw-count integration. Inspect the final figure and verify that both panels contain real supplied cells. Work only from the provided inputs; do not retrieve earlier TissueAgent or baseline reproductions."
ROLES = {'dataset_lohoff_et_al_seqfish.h5ad': 'seqfish_expression_reference', 'gastrulation_atlas_joint_umap.csv': 'atlas_joint_embedding', 'seqfish_author_joint_umap.csv': 'seqfish_joint_embedding', 'reference_lohoff-2a.png': 'reference', 'instructions_lohoff-2a.md': 'input_notes'}

def input_records():
    records = []
    for path in INPUTS:
        assert path.is_file(), f"Missing input: {path}"
        with path.open("rb") as stream:
            digest = hashlib.file_digest(stream, "sha256").hexdigest()
        records.append({"role": ROLES[path.name], "path": str(path), "name": path.name, "sha256": digest})
    return records

INPUT_RECORDS = input_records()
PROVENANCE = json.loads((DATA_DIR / "provenance_lohoff-2a.json").read_text())
for record in INPUT_RECORDS:
    if record["name"] in PROVENANCE["inputs"]:
        assert record["sha256"] == PROVENANCE["inputs"][record["name"]]["sha256"], record["name"]

print(json.dumps({
    "task": TASK_NAME,
    "datasets": [str(path) for path in DATASETS],
    "attachments": [str(path) for path in ATTACHMENTS],
    "supplied_colormap": False,
    "input_cells": PROVENANCE["input_cells"],
    "expected_plotted_cells": PROVENANCE["non_low_quality_cells"],
    "output_dir": str(OUTPUT_DIR),
}, indent=2))


## Run the CLI

Run this cell to start a fresh reproduction with the supplied atlas and seqFISH inputs. Set `OPENAI_API_KEY` in the notebook environment. The first run makes model calls; no completed benchmark result is bundled.

The CLI stages all three datasets, writes `transcript.log`, and retains execution state under `OUTPUT_DIR`. A later run reuses a completed local result only when the model, prompt, and input checksums match. Choose a new `OUTPUT_DIR` to rerun with changed inputs or settings.


In [ ]:
manifest_path = OUTPUT_DIR / "run_manifest.json"
metrics_path = OUTPUT_DIR / "metrics.json"
if OUTPUT_DIR.is_dir() and manifest_path.is_file() and metrics_path.is_file():
    manifest = json.loads(manifest_path.read_text())
    metrics = json.loads(metrics_path.read_text())
    terminal_state = metrics.get("outcome", {}).get("terminal_state")
    expected_hashes = {record["name"]: record["sha256"] for record in INPUT_RECORDS}
    cached_hashes = {record["name"]: record["sha256"] for record in manifest["inputs"]}
    assert terminal_state == "completed" and manifest.get("cli_exit_code") == 0
    assert manifest.get("supplied_colormap") is False
    assert manifest.get("prompt") == PROMPT, "Prompt changed; choose a new OUTPUT_DIR."
    assert manifest.get("model") == MODEL, "Model changed; choose a new OUTPUT_DIR."
    assert cached_hashes == expected_hashes, "Inputs changed; choose a new OUTPUT_DIR."
    print(json.dumps({
        "terminal_state": terminal_state,
        "elapsed_seconds": manifest["elapsed_seconds"],
        "output_dir": str(OUTPUT_DIR),
        "reused_completed_output": True,
    }, indent=2))
else:
    if OUTPUT_DIR.exists():
        raise RuntimeError(f"{OUTPUT_DIR} already exists. Choose a new OUTPUT_DIR for a fresh run.")
    assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in the notebook environment for a fresh run."
    OUTPUT_DIR.mkdir(parents=True)
    STATE_ROOT = OUTPUT_DIR / "state"
    with socket.socket() as sock:
        sock.bind(("127.0.0.1", 0))
        gateway_port = sock.getsockname()[1]
    env = os.environ.copy()
    env.update({
        "TISSUEAGENT_STATE_ROOT": str(STATE_ROOT),
        "TISSUEAGENT_GATEWAY_PORT": str(gateway_port),
        "TISSUEAGENT_ORCHESTRATION_REASONING_EFFORT": "medium",
        "TISSUEAGENT_WORKER_REASONING_EFFORT": "high",
        "NUMBA_CACHE_DIR": str(OUTPUT_DIR / "cache/numba"),
        "MPLCONFIGDIR": str(OUTPUT_DIR / "cache/matplotlib"),
        "XDG_CACHE_HOME": str(OUTPUT_DIR / "cache/xdg"),
        "IPYTHONDIR": str(OUTPUT_DIR / "cache/ipython"),
        "JUPYTER_RUNTIME_DIR": str(OUTPUT_DIR / "cache/jupyter_runtime"),
        "PYTHONPATH": str(REPO_ROOT / "src") + os.pathsep + str(REPO_ROOT),
        "PATH": str(Path(sys.executable).parent) + os.pathsep + env.get("PATH", ""),
    })
    for key in ["NUMBA_CACHE_DIR", "MPLCONFIGDIR", "XDG_CACHE_HOME", "IPYTHONDIR", "JUPYTER_RUNTIME_DIR"]:
        Path(env[key]).mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable, "-m", "cli", "--no-docker", "--json", "--model", MODEL,
        "--task-id", TASK_ID, "--seed", "0", "--attempt", "1",
        "--metrics-out", str(metrics_path),
    ]
    for dataset in DATASETS:
        command.extend(["--dataset", str(dataset)])
    for attachment in ATTACHMENTS:
        command.extend(["--attach", str(attachment)])
    command.append("-")

    transcript_path = OUTPUT_DIR / "transcript.log"
    (OUTPUT_DIR / "prompt.txt").write_text(PROMPT + "\n")
    started_at = datetime.now(timezone.utc)
    with transcript_path.open("w") as transcript:
        completed = subprocess.run(
            command, input=PROMPT, cwd=REPO_ROOT, env=env,
            stdout=transcript, stderr=subprocess.STDOUT, text=True, check=False,
        )
    finished_at = datetime.now(timezone.utc)
    project_dir = STATE_ROOT / "workspace/project"
    project_outputs = project_dir / "outputs"
    if project_outputs.is_dir():
        for source in project_outputs.iterdir():
            destination = OUTPUT_DIR / source.name
            if source.is_dir():
                shutil.copytree(source, destination, dirs_exist_ok=True)
            else:
                shutil.copy2(source, destination)
    if (project_dir / "metrics.json").is_file():
        shutil.copy2(project_dir / "metrics.json", OUTPUT_DIR / "project_metrics.json")
    if not metrics_path.is_file():
        raise RuntimeError(f"CLI exited {completed.returncode} without metrics. See {transcript_path}; state retained at {STATE_ROOT}.")
    metrics = json.loads(metrics_path.read_text())
    terminal_state = metrics.get("outcome", {}).get("terminal_state")
    manifest = {
        "task": TASK_NAME, "task_id": TASK_ID, "model": MODEL,
        "started_at_utc": started_at.isoformat(),
        "finished_at_utc": finished_at.isoformat(),
        "elapsed_seconds": metrics.get("run", {}).get("wall_time_s", (finished_at - started_at).total_seconds()),
        "elapsed_seconds_definition": "CLI-reported workflow wall time, falling back to subprocess wall time",
        "command": command, "prompt": PROMPT, "inputs": INPUT_RECORDS,
        "supplied_colormap": False, "cli_exit_code": completed.returncode,
        "terminal_state": terminal_state, "state_root": str(STATE_ROOT),
    }
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
    if completed.returncode != 0 or terminal_state != "completed":
        raise RuntimeError(f"CLI failed (exit={completed.returncode}, terminal_state={terminal_state}). See {transcript_path}; state retained at {STATE_ROOT}.")
    print(json.dumps({"terminal_state": terminal_state, "elapsed_seconds": manifest["elapsed_seconds"], "output_dir": str(OUTPUT_DIR)}, indent=2))


## Final figure

After the run, inspect the generated figure and reproduction report. The expected populations are **16,861 atlas cells and 52,568 seqFISH cells** after excluding `Low quality` cells. Verify these counts and preservation of the supplied coordinates in the generated artifacts; these are expected checks, not a precomputed result.


In [ ]:
preferred = [
    OUTPUT_DIR / "figures/lohoff_fig2a_recreated.png",
    OUTPUT_DIR / "figures/fig2a_umap_comparison.png",
    OUTPUT_DIR / "figures/lohoff_2a_umap.png",
    OUTPUT_DIR / "figures/umap_comparison.png",
]
selected = next((path for path in preferred if path.is_file()), None)
if selected is None:
    pngs = sorted(OUTPUT_DIR.rglob("*.png"))
    clean = [path for path in pngs if not re.search(r"attempt|comparison|compare|diff|reference|crop", path.name, re.IGNORECASE)]
    assert clean, f"No final figure found in {OUTPUT_DIR}"
    selected = max(clean, key=lambda path: path.stat().st_size)
print(selected.relative_to(REPO_ROOT))
display(Image(filename=str(selected)))
